# SQ-1 — Per-record MIA against the FL-released model

**Tests:** does the privacy card's `risk_calibration.attack_target.target_advantage` agree with the empirical worst-record TPR-at-low-FPR measured against the *FL-released model*?

**Headline metric:** worst-record TPR at FPR = 10⁻³ across a stratified per-record sweep [Carlini et al., S&P 2022].

**Threat profile:** R — semi-honest result consumer (queries the released model; no insider access).

**FL setup:** 50 pods × non-IID Dirichlet(α=0.5) slice of BloodMNIST; small MLP (2352 → 64 → 8); FedAvg + DP-SGD-style Gaussian noise; RDP accountant.

> Tutorial scale: 8 targets × 16 shadow runs → ~30–60 s end-to-end. Paper scale (`PAPER_SCALE = True`): 60 × 128 → ~15–30 min. **Power note:** the worst-record max is a high-variance statistic; below ~30 targets the estimate is dominated by sample variance, per Carlini et al. (2022) §4.

In [1]:
import json, sys, time
from pathlib import Path

import numpy as np

EVAL_DIR = Path.cwd().parents[1] if Path.cwd().name.startswith('sq') else Path.cwd()
sys.path.insert(0, str(EVAL_DIR))

from flta_eval import audit, attacks, datasets, fl

In [2]:
PAPER_SCALE = False
META_CLASSIFIER = "lira"      # "lira" (Carlini 2022 SOTA) or "logistic" (baseline)
MASTER_SEED = 20260525

# FL training config — small, fast, deterministic.
#
# Accountant note: σ=1.1, T=20, q=1.0, δ=1e-5 gives ε ≈ 44.6 under the
# single-α RDP envelope in flta_eval.fl.rdp_epsilon and ε ≈ 24.9 under the
# PRV accountant of Gopi et al. (NeurIPS 2021). See scripts/compare_accountants.py.
# The example card declares ε = 2.0 as a *design target*; the gap between
# declared and measured is what RISKCAL-002 is for.
fl_config = fl.FLConfig(
    n_rounds=20,
    client_lr=0.1,
    client_batch_size=32,
    clip_norm=1.0,
    noise_multiplier=1.1,
    secure_aggregation=False,
)

if PAPER_SCALE:
    n_targets, n_shadow_runs = 60, 128
else:
    n_targets, n_shadow_runs = 8, 16

ds = datasets.load_bloodmnist(EVAL_DIR / 'data')
manifest = datasets.load_partition(EVAL_DIR / 'data')
pod_data = [
    (ds['X_train'][np.array(manifest['pod_indices'][i])],
     ds['y_train'][np.array(manifest['pod_indices'][i])])
    for i in range(manifest['n_pods'])
]
print(f'FL pods: {len(pod_data)}; total samples: {sum(len(x) for x, _ in pod_data)}')
print(f'meta-classifier: {META_CLASSIFIER}')

FL pods: 50; total samples: 11956
meta-classifier: lira


In [3]:
t0 = time.time()
model = fl.MLP(input_dim=ds['input_dim'], hidden_dim=64, n_classes=ds['n_classes'])
model.init_from_seed(MASTER_SEED, 'fl.bloodmnist.v1')
trained, records = fl.federated_train(
    model=model, pod_data=pod_data,
    X_test=ds['X_test'], y_test=ds['y_test'],
    config=fl_config, master_seed=MASTER_SEED, namespace='fl.train.bloodmnist.v1',
)
train_elapsed = time.time() - t0
final_acc = trained.accuracy(ds['X_test'], ds['y_test'])
eps = fl.rdp_epsilon(noise_multiplier=fl_config.noise_multiplier,
                     n_rounds=fl_config.n_rounds,
                     sample_rate=1.0,
                     delta=1e-5)
print(f'FL training: {train_elapsed:.1f}s')
print(f'final test accuracy: {final_acc:.4f}')
print(f'RDP-accountant ε (δ=1e-5, σ={fl_config.noise_multiplier}, T={fl_config.n_rounds}): {eps:.2f}')

FL training: 0.8s
final test accuracy: 0.2122
RDP-accountant ε (δ=1e-5, σ=1.1, T=20): 44.57


In [4]:
t0 = time.time()
mia = attacks.per_record_mia(
    federated_model=trained, pod_data=pod_data,
    X_test=ds['X_test'], y_test=ds['y_test'],
    n_targets=n_targets, n_shadow_runs=n_shadow_runs,
    shadow_steps=20, shadow_batch_size=32,
    fpr_targets=(0.001, 0.01),
    meta_classifier=META_CLASSIFIER,
    n_bootstrap=1000,
    master_seed=MASTER_SEED, namespace='attacks.mia.bloodmnist.v1',
)
mia_elapsed = time.time() - t0
print(f'MIA per-record sweep ({META_CLASSIFIER}): {mia_elapsed:.1f}s ({mia.n_targets_swept} targets)')
print(f'worst-record TPR @ FPR=10⁻³: {mia.worst_record_tpr_at_fpr:.4f}  '
      f'95% CI [{mia.worst_record_tpr_ci_lower:.4f}, {mia.worst_record_tpr_ci_upper:.4f}]')
print(f'median TPR @ FPR=10⁻³     : {mia.median_tpr_at_fpr:.4f}')
print(f'worst target: pod {mia.worst_target_pod} index {mia.worst_target_index_in_pod}')
print()
print('Per-class stratification (worst record per class):')
for c in sorted(mia.per_class):
    s = mia.per_class[c]
    print(f"  class {c}: n={s['n_targets']:3d}  worst_TPR={s['worst_tpr']:.4f}  mean_TPR={s['mean_tpr']:.4f}")

# RMIA ablation — quick re-run with the same shadow budget for head-to-head.
rmia_t0 = time.time()
mia_rmia = attacks.per_record_mia(
    federated_model=trained, pod_data=pod_data,
    X_test=ds['X_test'], y_test=ds['y_test'],
    n_targets=n_targets, n_shadow_runs=n_shadow_runs,
    shadow_steps=20, shadow_batch_size=32,
    fpr_targets=(0.001, 0.01),
    meta_classifier='rmia',
    n_bootstrap=1000,
    master_seed=MASTER_SEED, namespace='attacks.mia.bloodmnist.rmia.v1',
)
print()
print(f'RMIA ablation [Zarifzadeh et al. ICML 2024] ({time.time()-rmia_t0:.1f}s):')
print(f'  worst-record TPR @ FPR=10⁻³: {mia_rmia.worst_record_tpr_at_fpr:.4f}  '
      f'95% CI [{mia_rmia.worst_record_tpr_ci_lower:.4f}, {mia_rmia.worst_record_tpr_ci_upper:.4f}]')
print(f'  vs LiRA worst-record:        {mia.worst_record_tpr_at_fpr:.4f}  '
      f'(RMIA {"tighter" if mia_rmia.worst_record_tpr_at_fpr > mia.worst_record_tpr_at_fpr else "looser"})')

MIA per-record sweep (lira): 1.1s (8 targets)
worst-record TPR @ FPR=10⁻³: 0.2468  95% CI [0.0000, 0.7233]
median TPR @ FPR=10⁻³     : 0.0332
worst target: pod 24 index 181

Per-class stratification (worst record per class):
  class 0: n=  1  worst_TPR=0.0000  mean_TPR=0.0000
  class 3: n=  3  worst_TPR=0.0451  mean_TPR=0.0226
  class 7: n=  4  worst_TPR=0.2468  mean_TPR=0.1192



RMIA ablation [Zarifzadeh et al. ICML 2024] (1.1s):
  worst-record TPR @ FPR=10⁻³: 0.2965  95% CI [0.0010, 0.3618]
  vs LiRA worst-record:        0.2468  (RMIA tighter)


In [5]:
record_path = audit.write_result_record(
    target_dir=EVAL_DIR / 'results' / 'sq1',
    attack='mia.per_record',
    variant='B-LiRA' if META_CLASSIFIER == 'lira' else 'B-logistic',
    threat_profile='R',
    dataset={'name': 'bloodmnist', 'sha256': manifest['npz_sha256'], 'n_train': manifest['n_train']},
    config={
        'paper_scale': PAPER_SCALE,
        'n_targets': n_targets, 'n_shadow_runs': n_shadow_runs,
        'meta_classifier': META_CLASSIFIER,
        'fl_rounds': fl_config.n_rounds, 'noise_multiplier': fl_config.noise_multiplier,
        'clip_norm': fl_config.clip_norm,
        'partition_alpha': manifest['partition_alpha'],
    },
    seed_namespace='attacks.mia.bloodmnist.v1',
    result={
        'worst_record_tpr_at_fpr_1e-3': mia.worst_record_tpr_at_fpr,
        'worst_record_tpr_ci_lower':    mia.worst_record_tpr_ci_lower,
        'worst_record_tpr_ci_upper':    mia.worst_record_tpr_ci_upper,
        'median_tpr_at_fpr_1e-3': mia.median_tpr_at_fpr,
        'fpr_target': mia.fpr_target,
        'n_targets_swept': mia.n_targets_swept,
        'fl_test_accuracy': final_acc,
        'rdp_accountant_epsilon': eps,
        'fl_training_elapsed_seconds': round(train_elapsed, 2),
        'mia_elapsed_seconds': round(mia_elapsed, 2),
        'per_class_worst_tpr': {str(k): v['worst_tpr'] for k, v in mia.per_class.items()},
        'per_class_mean_tpr':  {str(k): v['mean_tpr']  for k, v in mia.per_class.items()},
        'per_class_n_targets': {str(k): v['n_targets'] for k, v in mia.per_class.items()},
        # RMIA ablation
        'rmia_worst_record_tpr':    mia_rmia.worst_record_tpr_at_fpr,
        'rmia_worst_tpr_ci_lower':  mia_rmia.worst_record_tpr_ci_lower,
        'rmia_worst_tpr_ci_upper':  mia_rmia.worst_record_tpr_ci_upper,
    },
    tolerances={'absolute_tpr': 0.01, 'note': 'paper scale (60×128) target; tutorial scale is high-variance'},
)
print('Wrote', record_path.relative_to(EVAL_DIR))

Wrote results/sq1/mia_per_record__B-LiRA__2026-05-26T08-54-47Z.json


In [6]:
from flta_eval import rules

card = json.loads((EVAL_DIR / 'card' / 'example_chain.json').read_text())
target_advantage = card['risk_calibration']['attack_target']['target_advantage']
report = rules.evaluate_card(card, measured={'target_advantage': mia.worst_record_tpr_at_fpr})
print(f'target_advantage (card)  : {target_advantage:.4f}')
print(f'measured (this run)      : {mia.worst_record_tpr_at_fpr:.4f}')
print(f'delta (measured-target)  : {mia.worst_record_tpr_at_fpr - target_advantage:+.4f}')
print()
for f in report['findings']:
    print(f"  {f['severity']:5s} {f['rule_id']}: {f['title']}")

target_advantage (card)  : 0.0300
measured (this run)      : 0.2468
delta (measured-target)  : +0.2168

  AMBER RISKCAL-002: Measured advantage exceeds declared target


## Reading the result

The headline is the worst-record TPR at the operational FPR threshold (10⁻³), per Carlini *et al.* §3. At tutorial scale (8 targets × 16 shadow runs) the value is noisy — the max-over-8 estimator is high-variance, and the median is the more interpretable summary. Set `PAPER_SCALE = True` and re-run for the 60 × 128 configuration that converges.

**What this notebook surfaces that scalar reporting hides.** A reviewer reading only `(ε, AUROC)` would see something like `ε≈2.0, accuracy≈0.5`. The card's calibration block reports `worst-record TPR at FPR=10⁻³`, which is the operational quantity the card claims the deployment bounds — not derivable from ε alone, even with µ-DP conversion. The comparison loop above turns the calibration into a rule firing: if the measured advantage exceeds the declared target, `RISKCAL-002` fires AMBER. This is the rule engine's job.